In [0]:
from pyspark.sql import functions as F
import re
import unicodedata

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")


def normalize_name(value: str) -> str:
    value = value or ""
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.upper().strip()
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def normalize_columns(df):
    used = {}

    for old_name in df.columns:
        new_name = normalize_name(old_name)

        if new_name in used:
            used[new_name] += 1
            new_name = f"{new_name}_{used[new_name]}"
        else:
            used[new_name] = 1

        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    return df


def first_existing(df, candidates):
    normalized_candidates = [normalize_name(c) for c in candidates]

    for candidate in normalized_candidates:
        if candidate in df.columns:
            return F.col(candidate)

    return F.lit(None)


def clean_text(col):
    return F.trim(col.cast("string"))


def clean_upper(col):
    return F.upper(F.trim(col.cast("string")))


def clean_digits(col):
    return F.regexp_replace(col.cast("string"), r"[^0-9]", "")


def hash_key(*cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(c.cast("string"), F.lit("")) for c in cols]), 256)


In [0]:
firme_raw = normalize_columns(spark.table("company_ro.bronze.onrc_firme_raw"))
caen_raw = normalize_columns(spark.table("company_ro.bronze.onrc_caen_autorizat_raw"))
n_caen_raw = normalize_columns(spark.table("company_ro.bronze.n_caen_raw"))
stare_raw = normalize_columns(spark.table("company_ro.bronze.onrc_stare_firma_raw"))
n_stare_raw = normalize_columns(spark.table("company_ro.bronze.n_stare_firma_raw"))

print("Loaded bronze tables for ONRC dimensions")


In [0]:
stare_firma_bridge = (
    stare_raw
    .select(
        clean_text(first_existing(stare_raw, ["COD_INMATRICULARE", "NR_INMATRICULARE", "NUMAR_INMATRICULARE"])).alias("cod_inmatriculare"),
        clean_digits(first_existing(stare_raw, ["COD", "COD_STARE", "COD_STARE_FIRMA"])).alias("cod_stare_firma")
    )
    .filter(F.col("cod_inmatriculare").isNotNull())
    .filter(F.col("cod_inmatriculare") != "")
    .dropDuplicates(["cod_inmatriculare"])
)

firme_stage = (
    firme_raw
    .select(
        clean_text(first_existing(firme_raw, ["COD_INMATRICULARE", "NR_INMATRICULARE", "NUMAR_INMATRICULARE"])).alias("cod_inmatriculare"),
        clean_digits(first_existing(firme_raw, ["CUI", "COD_FISCAL", "CIF", "COD_UNIC_INREGISTRARE"])).alias("cui"),
        clean_text(first_existing(firme_raw, ["DENUMIRE", "DENUMIRE_FIRMA", "NUME_FIRMA"])).alias("denumire"),
        clean_upper(first_existing(firme_raw, ["JUDET", "DEN_JUDET", "DENUMIRE_JUDET", "ADR_JUDET", "ADR_DEN_JUDET", "ADRESA_JUDET"])).alias("judet"),
        clean_upper(first_existing(firme_raw, ["LOCALITATE", "DEN_LOCALITATE", "DENUMIRE_LOCALITATE", "ORAS", "MUNICIPIU", "ADR_LOCALITATE", "ADR_DEN_LOCALITATE", "ADRESA_LOCALITATE", "ADR_ORAS", "ADR_MUNICIPIU"])).alias("localitate"),
        clean_text(first_existing(firme_raw, ["ADRESA", "ADRESA_COMPLETA", "SEDIU", "ADR_ADRESA", "ADR_STRADA", "ADR_DEN_STRADA"])).alias("adresa"),
        clean_text(first_existing(firme_raw, ["FORMA_JURIDICA", "DEN_FORMA_JURIDICA", "FORM_JURIDICA", "COD_FORMA_JURIDICA"])).alias("forma_juridica"),
        first_existing(firme_raw, ["_ingested_at", "INGESTED_AT"]).alias("_ingested_at"),
        first_existing(firme_raw, ["_source_file", "SOURCE_FILE"]).alias("_source_file")
    )
    .filter(F.col("cod_inmatriculare").isNotNull())
    .filter(F.col("cod_inmatriculare") != "")
    .dropDuplicates(["cod_inmatriculare"])
    .join(stare_firma_bridge, "cod_inmatriculare", "left")
)

display(firme_stage.limit(20))


In [0]:
dim_stare_firma = (
    n_stare_raw
    .select(
        clean_digits(first_existing(n_stare_raw, ["COD", "COD_STARE", "COD_STARE_FIRMA"])).alias("cod_stare_firma"),
        clean_text(first_existing(n_stare_raw, ["DENUMIRE", "DENUMIRE_STARE", "STARE_FIRMA"])).alias("stare_firma")
    )
    .filter(F.col("cod_stare_firma").isNotNull())
    .filter(F.col("cod_stare_firma") != "")
    .dropDuplicates(["cod_stare_firma"])
    .withColumn(
        "is_activa",
        F.when(F.lower(F.col("stare_firma")) == "functiune", F.lit(True))
         .when(F.lower(F.col("stare_firma")) == "funcțiune", F.lit(True))
         .otherwise(F.lit(False))
    )
)

dim_forma_juridica = (
    firme_stage
    .select("forma_juridica")
    .filter(F.col("forma_juridica").isNotNull())
    .filter(F.col("forma_juridica") != "")
    .dropDuplicates(["forma_juridica"])
    .withColumn("forma_juridica_key", hash_key(F.col("forma_juridica")))
    .select("forma_juridica_key", "forma_juridica")
)

dim_localitate = (
    firme_stage
    .select("judet", "localitate")
    .filter(F.col("judet").isNotNull() | F.col("localitate").isNotNull())
    .dropDuplicates(["judet", "localitate"])
    .withColumn("localitate_key", hash_key(F.col("judet"), F.col("localitate")))
    .select("localitate_key", "judet", "localitate")
)

dim_adresa = (
    firme_stage
    .select("judet", "localitate", "adresa")
    .filter(F.col("adresa").isNotNull())
    .filter(F.col("adresa") != "")
    .dropDuplicates(["judet", "localitate", "adresa"])
    .withColumn("localitate_key", hash_key(F.col("judet"), F.col("localitate")))
    .withColumn("adresa_key", hash_key(F.col("localitate_key"), F.col("adresa")))
    .select("adresa_key", "localitate_key", "adresa")
)

dim_caen = (
    n_caen_raw
    .select(
        clean_digits(first_existing(n_caen_raw, ["COD_CAEN", "CAEN", "CLASA_CAEN", "COD"])).alias("cod_caen_raw"),
        clean_text(first_existing(n_caen_raw, ["DENUMIRE", "DENUMIRE_CAEN", "DESCRIERE", "DESCRIERE_CAEN"])).alias("denumire"),
        clean_text(first_existing(n_caen_raw, ["VERSIUNE_CAEN", "VER_CAEN", "VERSIUNE"])).alias("versiune_caen"),
        first_existing(n_caen_raw, ["_ingested_at", "INGESTED_AT"]).alias("_ingested_at"),
        first_existing(n_caen_raw, ["_source_file", "SOURCE_FILE"]).alias("_source_file")
    )
    .filter(F.length(F.col("cod_caen_raw")) == 4)
    .filter(F.col("denumire").isNotNull())
    .filter(F.col("denumire") != "")
    .withColumn("cod_caen", F.col("cod_caen_raw"))
    .dropDuplicates(["cod_caen"])
    .withColumn("caen_key", hash_key(F.col("cod_caen")))
    .select("caen_key", "cod_caen", "denumire", "versiune_caen", "_ingested_at", "_source_file")
)


In [0]:
dim_firma = (
    firme_stage
    .withColumn("firma_key", hash_key(F.col("cod_inmatriculare"), F.col("cui")))
    .withColumn("forma_juridica_key", F.when(F.col("forma_juridica").isNotNull() & (F.col("forma_juridica") != ""), hash_key(F.col("forma_juridica"))))
    .withColumn("adresa_localitate_key", F.when(F.col("judet").isNotNull() | F.col("localitate").isNotNull(), hash_key(F.col("judet"), F.col("localitate"))))
    .withColumn("adresa_key", F.when(F.col("adresa").isNotNull() & (F.col("adresa") != ""), hash_key(F.col("adresa_localitate_key"), F.col("adresa"))))
    .select(
        "firma_key",
        "cod_inmatriculare",
        "cui",
        "denumire",
        "cod_stare_firma",
        "forma_juridica_key",
        "adresa_key",
        "_ingested_at",
        "_source_file"
    )
)

bridge_firma_caen = (
    caen_raw
    .select(
        clean_text(first_existing(caen_raw, ["COD_INMATRICULARE", "NR_INMATRICULARE", "NUMAR_INMATRICULARE"])).alias("cod_inmatriculare"),
        clean_digits(first_existing(caen_raw, ["COD_CAEN_AUTORIZAT", "COD_CAEN", "CAEN"])).alias("cod_caen"),
        clean_text(first_existing(caen_raw, ["VER_CAEN_AUTORIZAT", "VERSIUNE_CAEN", "VER_CAEN"])).alias("versiune_caen"),
        first_existing(caen_raw, ["_ingested_at", "INGESTED_AT"]).alias("_ingested_at"),
        first_existing(caen_raw, ["_source_file", "SOURCE_FILE"]).alias("_source_file")
    )
    .filter(F.col("cod_inmatriculare").isNotNull())
    .filter(F.col("cod_caen").isNotNull())
    .filter(F.col("cod_caen") != "")
    .withColumn("cod_caen", F.lpad(F.col("cod_caen"), 4, "0"))
    .dropDuplicates(["cod_inmatriculare", "cod_caen", "versiune_caen"])
    .withColumn("firma_caen_key", hash_key(F.col("cod_inmatriculare"), F.col("cod_caen"), F.col("versiune_caen")))
    .select("firma_caen_key", "cod_inmatriculare", "cod_caen", "versiune_caen", "_ingested_at", "_source_file")
)

display(dim_firma.limit(20))


In [0]:
tables = {
    "dim_stare_firma": dim_stare_firma,
    "dim_forma_juridica": dim_forma_juridica,
    "dim_localitate": dim_localitate,
    "dim_adresa": dim_adresa,
    "dim_caen": dim_caen,
    "dim_firma": dim_firma,
    "bridge_firma_caen": bridge_firma_caen,
}

for table_name, df in tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"company_ro.silver.{table_name}")
    )
    print(f"Wrote company_ro.silver.{table_name}: {df.count():,} rows")


In [0]:
display(spark.sql("""
SELECT 'dim_firma' AS table_name, COUNT(*) AS rows FROM company_ro.silver.dim_firma
UNION ALL SELECT 'dim_stare_firma', COUNT(*) FROM company_ro.silver.dim_stare_firma
UNION ALL SELECT 'dim_forma_juridica', COUNT(*) FROM company_ro.silver.dim_forma_juridica
UNION ALL SELECT 'dim_localitate', COUNT(*) FROM company_ro.silver.dim_localitate
UNION ALL SELECT 'dim_adresa', COUNT(*) FROM company_ro.silver.dim_adresa
UNION ALL SELECT 'dim_caen', COUNT(*) FROM company_ro.silver.dim_caen
UNION ALL SELECT 'bridge_firma_caen', COUNT(*) FROM company_ro.silver.bridge_firma_caen
"""))

display(spark.sql("""
SELECT
  COUNT(*) AS companies,
  COUNT(s.cod_stare_firma) AS companies_matching_status,
  COUNT(fj.forma_juridica_key) AS companies_matching_legal_form,
  COUNT(a.adresa_key) AS companies_matching_address,
  COUNT(l.localitate_key) AS companies_matching_location_via_address
FROM company_ro.silver.dim_firma f
LEFT JOIN company_ro.silver.dim_stare_firma s ON f.cod_stare_firma = s.cod_stare_firma
LEFT JOIN company_ro.silver.dim_forma_juridica fj ON f.forma_juridica_key = fj.forma_juridica_key
LEFT JOIN company_ro.silver.dim_adresa a ON f.adresa_key = a.adresa_key
LEFT JOIN company_ro.silver.dim_localitate l ON a.localitate_key = l.localitate_key
"""))

display(spark.sql("""
SELECT
  COUNT(*) AS bridge_rows,
  COUNT(f.cod_inmatriculare) AS rows_matching_dim_firma,
  COUNT(c.cod_caen) AS rows_matching_dim_caen
FROM company_ro.silver.bridge_firma_caen b
LEFT JOIN company_ro.silver.dim_firma f ON b.cod_inmatriculare = f.cod_inmatriculare
LEFT JOIN company_ro.silver.dim_caen c ON b.cod_caen = c.cod_caen
"""))
